# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [3]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/netflix_catalogue.csv')

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Filter to most common ratings only
ratings_to_keep = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df[df['rating'].isin(ratings_to_keep)]

# Build pivot table: rows = rating, columns = decade, values = count
heatmap_data = (
    df_filtered
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
    .pivot(index='rating', columns='decade', values='count')
    .fillna(0)
    .astype(int)
)

# Plot
fig = px.imshow(
    heatmap_data,
    color_continuous_scale='Blues',
    text_auto=True,
    labels={'x': 'Decade', 'y': 'Content Rating', 'color': 'Titles'},
    title='TV-MA Dominates Recent Decades While PG Led the 1980s–1990s',
)

fig.update_layout(
    xaxis_title='Decade',
    yaxis_title='Content Rating',
    coloraxis_colorbar_title='# Titles',
)

fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [11]:
# Task 2
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
df.columns = df.columns.str.strip()

movies = df[df['type'] == 'Movie'].copy()

# Use release_year since added_year has no growth pattern in your dataset
yearly = (
    movies[movies['release_year'].between(2015, 2023)]
    .groupby('release_year')
    .size()
    .reset_index(name='count')
    .sort_values('release_year')
)

years  = yearly['release_year'].astype(str).tolist()
counts = yearly['count'].tolist()
total  = sum(counts)

# Running base for waterfall effect (invisible bottom bars)
running, bases = 0, []
for c in counts:
    bases.append(running)
    running += c

peak_idx   = counts.index(max(counts))
peak_year  = years[peak_idx]
peak_count = counts[peak_idx]

fig = go.Figure()

# Invisible base bars
fig.add_trace(go.Bar(
    x=years + ['Total'],
    y=bases + [0],
    marker_color='rgba(0,0,0,0)',
    hoverinfo='skip',
    showlegend=False,
))

# Visible addition bars
fig.add_trace(go.Bar(
    x=years + ['Total'],
    y=counts + [total],
    marker_color=['#2ecc71'] * len(years) + ['#2980b9'],
    text=[f'{v:,}' for v in counts + [total]],
    textposition='outside',
    name='Movies',
    showlegend=False,
))

fig.add_annotation(
    x=4, y=bases[peak_idx] + peak_count,
    text=f'▲ Peak<br><b>{peak_count:,} titles</b>',
    showarrow=True, arrowhead=2,
    arrowcolor='#e74c3c',
    font={'color': '#e74c3c', 'size': 12},
    bgcolor='rgba(255,255,255,0.9)',
    bordercolor='#e74c3c', borderwidth=1.5,
    ax=55, ay=-55,
)

fig.update_layout(
    barmode='stack',
    title={'text': f'Netflix Movie Releases Peaked in {peak_year} with {peak_count:,} Titles',
           'font': {'size': 15}, 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Release Year',
    yaxis_title='Movies',
    xaxis={'type': 'category', 'showgrid': False},
    plot_bgcolor='#f9f9f9',
    paper_bgcolor='white',
    yaxis={'gridcolor': '#ddd', 'zeroline': False},
    margin={'t': 90, 'b': 60, 'l': 60, 'r': 40},
    height=520, width=820,
)

fig.show()